In [1]:
import os
import pandas as pd
import shutil
import pydicom
import numpy as np
import zipfile

# Script for splitting DWI sequences for which all b values are lumped into one

dicomDir = '/Volumes/T7/regeneron_radiomics/split_b_test'

patients = sorted([x for x in os.listdir(dicomDir) if x[0] != '.'])

# For this folder, data are arranged as patient-series-images, so there is no study name

for patient in patients:
    #os.mkdir(os.path.join(outDir,patient))
    seriess = sorted([x for x in os.listdir(os.path.join(dicomDir,patient)) if x[0] != '.'])
                            
    for series in seriess:
        #os.mkdir(os.path.join(outDir,patient,series))
        series_path = os.path.join(dicomDir,patient,series)
        images = sorted([x for x in os.listdir(os.path.join(dicomDir,patient,series)) if x[0] != '.'])
        
        for image in images:        
            image_path = os.path.join(series_path,image) 
            ds = pydicom.filereader.dcmread(image_path)
            
            # Check if image belongs to a DWI series. If it does, create a new folder labeled by the corresponding b-value
            # and move it there
            
            if 'SequenceName' in ds and 'ep_' in ds.SequenceName:    
                bval_dir = os.path.join(dicomDir,patient,series+ds.SequenceName) 
                if os.path.exists(bval_dir) == False:            
                    os.mkdir(bval_dir)
                    shutil.move(image_path,bval_dir)
                else:
                    shutil.move(image_path,bval_dir)
                
            elif (0x0018, 0x9087) in ds:
                bval = ds[(0x0018, 0x9087)].value
                bval_dir = os.path.join(dicomDir,patient,series+'_'+str(bval))
                if os.path.exists(bval_dir) == False:            
                    os.mkdir(bval_dir)
                    shutil.move(image_path,bval_dir)
                else:
                    shutil.move(image_path,bval_dir)
                    
# Remove the older folder with all b-values lumped together, which should now be empty

patients = sorted([x for x in os.listdir(dicomDir) if x[0] != '.'])
for patient in patients:
    seriess = sorted([x for x in os.listdir(os.path.join(dicomDir,patient)) if x[0] != '.'])
    for series in seriess:     
        old_series_path = os.path.join(dicomDir,patient,series)
        if not os.listdir(old_series_path):
            try:       
                os.rmdir(old_series_path)
            except OSError as e:
                print(f"Error removing folder '{old_series_path}': {e}")            